# 🚀 TravelMate AI - 18 FastAPI Multi-City Backend

Your current browser screenshot is still showing the old **Manali-only** application.

Notebook 17 created the reusable `src/` engine. This notebook replaces the old backend with a city-aware FastAPI application.

```text
Streamlit
   ↓
FastAPI
   ↓
src/recommender.py
   ↓
src/itinerary.py
   ↓
Multi-city data
```

Supported destinations come directly from the dataset, so the API does not hard-code a false Goa/Manali distinction.


## 1. Project paths

In [6]:
from pathlib import Path
import pandas as pd
import sys

PROJECT_ROOT = Path("..").resolve()
APP_DIR = PROJECT_ROOT / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("App directory:", APP_DIR)


Project root: D:\college_work\PG\linkedIn_projects\TravelMate-AI
App directory: D:\college_work\PG\linkedIn_projects\TravelMate-AI\app


## 2. Verify shared engine

In [7]:
required_files = [
    "data_loader.py",
    "recommender.py",
    "itinerary.py",
    "utils.py",
    "__init__.py",
]

missing = [
    f for f in required_files
    if not (PROJECT_ROOT / "src" / f).exists()
]

if missing:
    raise FileNotFoundError(
        f"Missing src files: {missing}. Run Notebook 17 first."
    )

print("✅ Shared engine verified")


✅ Shared engine verified


## 3. Write the new `app/main.py`

In [8]:
main_code = 'from __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\n\nfrom src.data_loader import load_places, available_cities, load_embeddings\nfrom src.recommender import TravelRecommender\nfrom src.itinerary import generate_itinerary\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\n\nDATA_PATH = (\n    PROJECT_ROOT\n    / "data"\n    / "processed"\n    / "multi_city_recommendation_features.csv"\n)\n\nEMBEDDING_PATH = (\n    PROJECT_ROOT\n    / "data"\n    / "embeddings"\n    / "multi_city_place_embeddings.npy"\n)\n\nMAX_DAY_MINUTES = 7 * 60\nDEFAULT_DAYS = 3\nDEFAULT_TOP_N = 5\n\n\nplaces = load_places(DATA_PATH)\n\nif EMBEDDING_PATH.exists():\n    embeddings = load_embeddings(EMBEDDING_PATH)\nelse:\n    embeddings = None\n\nrecommender = TravelRecommender(\n    df=places,\n    place_embeddings=embeddings,\n)\n\n\ndef infer_activity_type(text: str) -> str:\n    text = str(text).lower()\n\n    if "waterfall" in text or "falls" in text:\n        return "waterfall"\n    if "rafting" in text:\n        return "rafting"\n    if "trek" in text:\n        return "trekking"\n    if "viewpoint" in text or "view point" in text:\n        return "viewpoint"\n    if any(x in text for x in ["temple", "church", "mosque", "gurudwara", "monastery"]):\n        return "religious_site"\n    if any(x in text for x in ["fort", "palace", "museum", "heritage", "castle"]):\n        return "heritage"\n    if any(x in text for x in ["market", "bazaar", "mall", "shopping"]):\n        return "shopping"\n    if any(x in text for x in ["beach", "lake", "river", "park", "forest", "garden"]):\n        return "nature"\n    if "snow" in text or "ski" in text:\n        return "winter_experience"\n\n    return "sightseeing"\n\n\ndef estimate_visit_minutes(activity_type: str) -> int:\n    return {\n        "waterfall": 90,\n        "rafting": 120,\n        "trekking": 150,\n        "viewpoint": 45,\n        "religious_site": 60,\n        "heritage": 120,\n        "shopping": 90,\n        "nature": 90,\n        "winter_experience": 90,\n        "sightseeing": 60,\n    }.get(activity_type, 60)\n\n\ndef estimate_price_level(text: str) -> int:\n    text = str(text).lower()\n\n    if "rafting" in text or "ski" in text or "adventure" in text:\n        return 3\n\n    if any(x in text for x in ["shopping", "market", "bazaar", "mall"]):\n        return 2\n\n    return 1\n\n\ndef add_planning_metadata(candidates: pd.DataFrame) -> pd.DataFrame:\n    result = candidates.copy()\n\n    planning_text = (\n        result["name"].fillna("").astype(str)\n        + " "\n        + result["category"].fillna("").astype(str)\n    )\n\n    result["activity_type"] = planning_text.apply(\n        infer_activity_type\n    )\n\n    result["estimated_visit_minutes"] = (\n        result["activity_type"].apply(\n            estimate_visit_minutes\n        )\n    )\n\n    result["estimated_price_level"] = (\n        planning_text.apply(\n            estimate_price_level\n        )\n    )\n\n    return result\n\n\nclass Preferences(BaseModel):\n    nature: float = Field(0.5, ge=0.0, le=1.0)\n    history: float = Field(0.3, ge=0.0, le=1.0)\n    culture: float = Field(0.4, ge=0.0, le=1.0)\n    adventure: float = Field(0.3, ge=0.0, le=1.0)\n    photography: float = Field(0.5, ge=0.0, le=1.0)\n    shopping: float = Field(0.2, ge=0.0, le=1.0)\n    religious: float = Field(0.2, ge=0.0, le=1.0)\n    family: float = Field(0.4, ge=0.0, le=1.0)\n\n\nclass TravelRequest(BaseModel):\n    destination: str = Field(..., min_length=1)\n    query: str = Field(..., min_length=1)\n    days: int = Field(DEFAULT_DAYS, ge=1, le=14)\n    top_n: int = Field(DEFAULT_TOP_N, ge=1, le=15)\n    preferences: Preferences\n\n\napp = FastAPI(\n    title="TravelMate AI API",\n    description="City-aware AI travel recommendation and itinerary backend.",\n    version="2.0.0",\n)\n\n\n@app.get("/")\ndef root():\n    return {\n        "app": "TravelMate AI",\n        "version": "2.0.0",\n        "status": "running",\n        "supported_cities": available_cities(places),\n    }\n\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "healthy",\n        "places_loaded": len(places),\n        "cities_loaded": len(available_cities(places)),\n        "embedding_rows": len(recommender.place_embeddings),\n    }\n\n\n@app.get("/cities")\ndef cities():\n    return {\n        "cities": available_cities(places)\n    }\n\n\n@app.post("/recommend")\ndef recommend(request: TravelRequest):\n    try:\n        result = recommender.recommend(\n            destination=request.destination,\n            query=request.query,\n            user_preferences=request.preferences.model_dump(),\n            top_n=request.top_n,\n        )\n\n        result = add_planning_metadata(result)\n\n        columns = [\n            "name",\n            "city",\n            "category",\n            "rating",\n            "reviews",\n            "travel_tags",\n            "activity_type",\n            "estimated_visit_minutes",\n            "estimated_price_level",\n            "structured_score",\n            "tfidf_score",\n            "semantic_score",\n            "final_score",\n            "latitude",\n            "longitude",\n        ]\n\n        result = result[\n            [c for c in columns if c in result.columns]\n        ]\n\n        return {\n            "destination": request.destination,\n            "query": request.query,\n            "count": len(result),\n            "recommendations": result.to_dict(\n                orient="records"\n            ),\n        }\n\n    except ValueError as exc:\n        raise HTTPException(\n            status_code=400,\n            detail=str(exc),\n        ) from exc\n\n\n@app.post("/itinerary")\ndef itinerary(request: TravelRequest):\n    try:\n        candidate_count = min(\n            max(request.top_n, request.days * 4),\n            len(places),\n        )\n\n        candidates = recommender.recommend(\n            destination=request.destination,\n            query=request.query,\n            user_preferences=request.preferences.model_dump(),\n            top_n=candidate_count,\n        )\n\n        candidates = add_planning_metadata(candidates)\n\n        plan = generate_itinerary(\n            candidates=candidates,\n            days=request.days,\n            max_day_minutes=MAX_DAY_MINUTES,\n            average_speed_kmph=25,\n        )\n\n        return {\n            "destination": request.destination,\n            "query": request.query,\n            "days_requested": request.days,\n            "scheduled_stops": len(plan),\n            "daily_budget_minutes": MAX_DAY_MINUTES,\n            "itinerary": plan.to_dict(\n                orient="records"\n            ),\n        }\n\n    except ValueError as exc:\n        raise HTTPException(\n            status_code=400,\n            detail=str(exc),\n        ) from exc\n\n    except RuntimeError as exc:\n        raise HTTPException(\n            status_code=500,\n            detail=str(exc),\n        ) from exc\n'
main_path = APP_DIR / "main.py"
main_path.write_text(main_code, encoding="utf-8")

print(f"✅ Created: {main_path}")


✅ Created: D:\college_work\PG\linkedIn_projects\TravelMate-AI\app\main.py


## 4. Import the backend

In [9]:
from app.main import (
    app,
    places,
    recommender,
    available_cities,
)

print("✅ FastAPI module imported")
print("Places:", len(places))
print("Cities:", sorted(places["city"].unique()))


✅ FastAPI module imported
Places: 120
Cities: ['Goa', 'Jaipur', 'Manali', 'Rishikesh', 'Shimla', 'Udaipur']


## 5. Test the API logic directly

In [10]:
from app.main import (
    TravelRequest,
    Preferences,
    recommend,
    itinerary,
)

request = TravelRequest(
    destination="Goa",
    query="beautiful beaches and relaxing scenic places",
    days=3,
    top_n=5,
    preferences=Preferences(
        nature=1.0,
        history=0.1,
        culture=0.4,
        adventure=0.4,
        photography=1.0,
        shopping=0.2,
        religious=0.0,
        family=0.5,
    ),
)

recommendation_response = recommend(request)

pd.DataFrame(
    recommendation_response["recommendations"]
)


,name,city,category,rating,reviews,travel_tags,activity_type,estimated_visit_minutes,estimated_price_level,structured_score,tfidf_score,semantic_score,final_score,latitude,longitude
0,"Calangute Beach, Goa",Goa,Tourist attraction,4.3,103884,"nature, photography, family",nature,90,1,0.891720,0.392560,0.830408,0.718633,15.544721,73.754669
1,Keri Beach,Goa,Tourist attraction,4.6,5070,"nature, photography, family",nature,90,1,0.891720,0.221955,0.815655,0.662580,15.708774,73.692984
2,Kuske Waterfall,Goa,Tourist attraction,4.4,332,"nature, photography",waterfall,90,1,0.873704,0.243288,0.726848,0.592479,15.021708,74.208660
3,Dudhsagar Falls,Goa,Tourist attraction,4.6,32335,nature,waterfall,90,1,0.617802,0.244277,0.709716,0.587564,15.314438,74.314307
4,Twin Waterfall,Goa,Tourist attraction,4.0,186,"nature, photography",waterfall,90,1,0.873704,0.243288,0.763477,0.566573,15.361667,74.042960


In [11]:
goa_result = pd.DataFrame(
    recommendation_response["recommendations"]
)

assert not goa_result.empty
assert goa_result["city"].eq("Goa").all()

print("✅ Goa recommendations are city-specific")


✅ Goa recommendations are city-specific


## 6. Test Goa itinerary

In [12]:
goa_itinerary_response = itinerary(request)

goa_itinerary = pd.DataFrame(
    goa_itinerary_response["itinerary"]
)

goa_itinerary


,city,day,stop,place,activity_type,arrival,departure,travel_before_minutes,visit_minutes,estimated_price_level,final_score,latitude,longitude
0,Goa,1,1,"Calangute Beach, Goa",nature,9:00 AM,10:30 AM,0.00,90,1,0.7186,15.544721,73.754669
1,Goa,1,2,Sinquerim Fort,heritage,10:43 AM,12:43 PM,12.71,120,1,0.5537,15.498466,73.766404
2,Goa,1,3,Fort Aguada,heritage,12:45 PM,2:45 PM,2.51,120,1,0.5401,15.492252,73.773746
3,Goa,2,1,Keri Beach,nature,9:00 AM,10:30 AM,0.00,90,1,0.6626,15.708774,73.692984
4,Goa,2,2,Chapora Fort,heritage,11:00 AM,1:00 PM,30.00,120,1,0.4918,15.604638,73.736963
5,Goa,2,3,Kesarval Spring Verna Waterfall,waterfall,2:17 PM,3:47 PM,77.16,90,1,0.5248,15.382312,73.928801
6,Goa,3,1,Kuske Waterfall,waterfall,9:00 AM,10:30 AM,0.00,90,1,0.5925,15.021708,74.208660
7,Goa,3,2,Cabo de Rama Fort South Goa,heritage,11:46 AM,1:46 PM,76.11,120,1,0.5660,15.088785,73.921593
8,Goa,3,3,Sunset View Point Colva,viewpoint,2:36 PM,3:21 PM,49.70,45,1,0.5285,15.274854,73.913585


In [13]:
assert not goa_itinerary.empty
assert goa_itinerary["city"].eq("Goa").all()

daily = (
    goa_itinerary
    .assign(
        total_minutes=lambda x:
            x["travel_before_minutes"]
            + x["visit_minutes"]
    )
    .groupby("day")["total_minutes"]
    .sum()
)

assert (daily <= 7 * 60 + 1e-9).all()

print("✅ Goa itinerary passed city + time checks")


✅ Goa itinerary passed city + time checks


## 7. Test every supported city

In [14]:
test_rows = []

for city in sorted(places["city"].unique()):

    request = TravelRequest(
        destination=city,
        query="best places for a memorable trip",
        days=3,
        top_n=5,
        preferences=Preferences()
    )

    rec = recommend(request)
    rec_df = pd.DataFrame(
        rec["recommendations"]
    )

    test_rows.append({
        "city": city,
        "recommendations": len(rec_df),
        "city_correct": (
            not rec_df.empty
            and rec_df["city"].eq(city).all()
        ),
    })

api_tests = pd.DataFrame(test_rows)
api_tests


,city,recommendations,city_correct
0,Goa,5,True
1,Jaipur,5,True
2,Manali,5,True
3,Rishikesh,5,True
4,Shimla,5,True
5,Udaipur,5,True


In [15]:
assert api_tests["city_correct"].all()

print(
    "✅ All supported cities passed API city validation."
)


✅ All supported cities passed API city validation.


# 🚀 Run the backend

From the **TravelMate-AI project root**:

```powershell
uvicorn app.main:app --reload
```

Then open:

```text
http://127.0.0.1:8000/docs
```

Test:

```text
GET /cities
GET /health
POST /recommend
POST /itinerary
```

### Important

Your old Streamlit app will continue showing the old Manali-only warning until we update the frontend to use this backend.

That is the **next step** after this notebook passes.
